<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/10_Cross_Dataset_Robustness_%2B_Statistical_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK 10.0 — ENVIRONMENT & CONFIGURATION
# ============================================================

from pathlib import Path
import json
import hashlib
import warnings
import gc
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10")
print("CROSS-DATASET ROBUSTNESS + STATISTICAL ANALYSIS")
print("=" * 100)


# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

from google.colab import drive

DRIVE_ROOT = Path("/content/drive")

if not DRIVE_ROOT.exists():
    drive.mount(str(DRIVE_ROOT))

MYDRIVE = DRIVE_ROOT / "MyDrive"

if not MYDRIVE.exists():
    raise FileNotFoundError(
        f"MyDrive not found:\n{MYDRIVE}"
    )


# ------------------------------------------------------------
# Project
# ------------------------------------------------------------

PROJECT_ROOT = MYDRIVE / "AIR_LLM_Research"

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"AIR_LLM_Research not found:\n{PROJECT_ROOT}"
    )


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

MECHANISMS = [
    "MCAR",
    "MAR",
    "MNAR_APPROXIMATION"
]

MISSINGNESS_RATES = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

REPETITIONS = 5

MASTER_SEED = 42


# ------------------------------------------------------------
# Notebook directories
# ------------------------------------------------------------

NB07_DIR = PROJECT_ROOT / "data" / "notebook_07"
NB08_DIR = PROJECT_ROOT / "data" / "notebook_08"
NB09_DIR = PROJECT_ROOT / "data" / "notebook_09"
NB10_DIR = PROJECT_ROOT / "data" / "notebook_10"


RESULTS_DIR = NB10_DIR / "results"
ROBUSTNESS_DIR = NB10_DIR / "robustness"
STATISTICS_DIR = NB10_DIR / "statistics"
TABLES_DIR = NB10_DIR / "tables"
FIGURES_DIR = NB10_DIR / "figures"
METADATA_DIR = NB10_DIR / "metadata"


for directory in [
    NB10_DIR,
    RESULTS_DIR,
    ROBUSTNESS_DIR,
    STATISTICS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    METADATA_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


MANIFEST_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "notebook_10_manifest.json"
)

MANIFEST_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)


print("\nPROJECT")
print("-" * 100)
print(f"Project root : {PROJECT_ROOT}")
print(f"Datasets     : {len(DATASETS)}")
print(f"Mechanisms   : {len(MECHANISMS)}")
print(f"Rates        : {len(MISSINGNESS_RATES)}")
print(f"Repetitions  : {REPETITIONS}")

print("\nNOTEBOOK 10 DIRECTORIES")
print("-" * 100)

print(f"Results      : {RESULTS_DIR}")
print(f"Robustness   : {ROBUSTNESS_DIR}")
print(f"Statistics   : {STATISTICS_DIR}")
print(f"Tables       : {TABLES_DIR}")
print(f"Figures      : {FIGURES_DIR}")

In [ ]:
# ============================================================
# NOTEBOOK 10.0 — ENVIRONMENT & CONFIGURATION
# ============================================================

from pathlib import Path
import json
import hashlib
import warnings
import gc
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10")
print("CROSS-DATASET ROBUSTNESS + STATISTICAL ANALYSIS")
print("=" * 100)


# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

from google.colab import drive

DRIVE_ROOT = Path("/content/drive")

if not DRIVE_ROOT.exists():
    drive.mount(str(DRIVE_ROOT))

MYDRIVE = DRIVE_ROOT / "MyDrive"

if not MYDRIVE.exists():
    raise FileNotFoundError(
        f"MyDrive not found:\n{MYDRIVE}"
    )


# ------------------------------------------------------------
# Project
# ------------------------------------------------------------

PROJECT_ROOT = MYDRIVE / "AIR_LLM_Research"

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"AIR_LLM_Research not found:\n{PROJECT_ROOT}"
    )


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

MECHANISMS = [
    "MCAR",
    "MAR",
    "MNAR_APPROXIMATION"
]

MISSINGNESS_RATES = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

REPETITIONS = 5

MASTER_SEED = 42


# ------------------------------------------------------------
# Notebook directories
# ------------------------------------------------------------

NB07_DIR = PROJECT_ROOT / "data" / "notebook_07"
NB08_DIR = PROJECT_ROOT / "data" / "notebook_08"
NB09_DIR = PROJECT_ROOT / "data" / "notebook_09"
NB10_DIR = PROJECT_ROOT / "data" / "notebook_10"


RESULTS_DIR = NB10_DIR / "results"
ROBUSTNESS_DIR = NB10_DIR / "robustness"
STATISTICS_DIR = NB10_DIR / "statistics"
TABLES_DIR = NB10_DIR / "tables"
FIGURES_DIR = NB10_DIR / "figures"
METADATA_DIR = NB10_DIR / "metadata"


for directory in [
    NB10_DIR,
    RESULTS_DIR,
    ROBUSTNESS_DIR,
    STATISTICS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    METADATA_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


MANIFEST_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "notebook_10_manifest.json"
)

MANIFEST_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)


print("\nPROJECT")
print("-" * 100)
print(f"Project root : {PROJECT_ROOT}")
print(f"Datasets     : {len(DATASETS)}")
print(f"Mechanisms   : {len(MECHANISMS)}")
print(f"Rates        : {len(MISSINGNESS_RATES)}")
print(f"Repetitions  : {REPETITIONS}")

print("\nNOTEBOOK 10 DIRECTORIES")
print("-" * 100)

print(f"Results      : {RESULTS_DIR}")
print(f"Robustness   : {ROBUSTNESS_DIR}")
print(f"Statistics   : {STATISTICS_DIR}")
print(f"Tables       : {TABLES_DIR}")
print(f"Figures      : {FIGURES_DIR}")

In [ ]:
# ============================================================
# NOTEBOOK 10.1 — ARTIFACT DISCOVERY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.1")
print("PREVIOUS NOTEBOOK ARTIFACT DISCOVERY")
print("=" * 100)


def discover_files(root, keywords):
    """
    Recursively discover files whose names contain
    at least one of the supplied keywords.
    """

    found = []

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        name = path.name.lower()

        if any(
            keyword.lower() in name
            for keyword in keywords
        ):
            found.append(path)

    return sorted(found)


NB07_FILES = discover_files(
    NB07_DIR,
    [
        "evaluation",
        "candidate",
        "result",
        "utility"
    ]
) if NB07_DIR.exists() else []


NB08_FILES = discover_files(
    NB08_DIR,
    [
        "confidence",
        "abstention",
        "selection",
        "result"
    ]
) if NB08_DIR.exists() else []


NB09_FILES = discover_files(
    NB09_DIR,
    [
        "explanation",
        "explainability",
        "decision",
        "result"
    ]
) if NB09_DIR.exists() else []


print("\nNOTEBOOK 07 FILES")
print("-" * 100)

for path in NB07_FILES[:50]:
    print(path)


print("\nNOTEBOOK 08 FILES")
print("-" * 100)

for path in NB08_FILES[:50]:
    print(path)


print("\nNOTEBOOK 09 FILES")
print("-" * 100)

for path in NB09_FILES[:50]:
    print(path)


print("\nDISCOVERY SUMMARY")
print("-" * 100)

print(f"Notebook 07 files : {len(NB07_FILES)}")
print(f"Notebook 08 files : {len(NB08_FILES)}")
print(f"Notebook 09 files : {len(NB09_FILES)}")

In [ ]:
# ============================================================
# NOTEBOOK 10.2 — LOAD EXPERIMENTAL RESULTS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.2")
print("LOADING EXPERIMENTAL RESULTS")
print("=" * 100)


ALL_RESULT_FILES = (
    NB07_FILES +
    NB08_FILES +
    NB09_FILES
)


CSV_FILES = [
    p for p in ALL_RESULT_FILES
    if p.suffix.lower() == ".csv"
]


RESULT_TABLES = {}


for path in CSV_FILES:

    try:

        df = pd.read_csv(path)

        if len(df) == 0:
            continue

        RESULT_TABLES[str(path)] = df

        print(
            f"LOADED | "
            f"{path.name:45s} | "
            f"rows={len(df):8d} | "
            f"cols={len(df.columns):4d}"
        )

    except Exception as exc:

        print(
            f"SKIPPED | {path.name} | {exc}"
        )


print("\nLoaded tables:", len(RESULT_TABLES))

In [ ]:
# ============================================================
# NOTEBOOK 10.3 — IDENTIFY MAIN EVALUATION TABLE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.3")
print("IDENTIFYING MAIN EVALUATION TABLE")
print("=" * 100)


REQUIRED_RESULT_COLUMNS = [
    "dataset_id",
    "mechanism",
    "requested_rate",
    "repetition"
]


def score_result_table(df):

    score = 0

    columns = set(
        c.lower()
        for c in df.columns
    )

    for column in REQUIRED_RESULT_COLUMNS:

        if column in columns:
            score += 5

    metric_keywords = [
        "mae",
        "rmse",
        "utility",
        "accuracy",
        "f1",
        "wasserstein",
        "runtime",
        "strategy"
    ]

    for keyword in metric_keywords:

        if any(
            keyword in c
            for c in columns
        ):
            score += 1

    return score


ranked_tables = []

for path, df in RESULT_TABLES.items():

    ranked_tables.append(
        (
            score_result_table(df),
            path,
            df
        )
    )


ranked_tables.sort(
    key=lambda x: x[0],
    reverse=True
)


if not ranked_tables:

    raise RuntimeError(
        "No experimental result tables were found.\n"
        "Notebook 10 requires outputs from the earlier "
        "evaluation/selection notebooks."
    )


RESULT_SCORE, RESULT_PATH, EVALUATION_DF = (
    ranked_tables[0]
)


print("\nSELECTED RESULT TABLE")
print("-" * 100)

print(f"File  : {RESULT_PATH}")
print(f"Score : {RESULT_SCORE}")
print(f"Rows  : {len(EVALUATION_DF)}")
print(f"Cols  : {len(EVALUATION_DF.columns)}")


display(
    EVALUATION_DF.head()
)

In [ ]:
# ============================================================
# NOTEBOOK 10.5 — METRIC DISCOVERY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.5")
print("METRIC DISCOVERY")
print("=" * 100)


NUMERIC_COLUMNS = [
    c for c in RESULTS.columns
    if pd.api.types.is_numeric_dtype(
        RESULTS[c]
    )
]


METRIC_PATTERNS = {

    "accuracy":
        ["mae", "rmse", "r2", "accuracy",
         "macro_f1", "weighted_f1"],

    "distribution":
        ["wasserstein", "js", "kl",
         "distribution"],

    "dependency":
        ["pearson", "spearman",
         "mutual_information",
         "dependency"],

    "predictive":
        ["predictive", "downstream",
         "auc", "f1"],

    "efficiency":
        ["runtime", "time", "memory",
         "cost"],

    "utility":
        ["utility", "score"]
}


METRIC_COLUMNS = {}


for group, patterns in METRIC_PATTERNS.items():

    METRIC_COLUMNS[group] = [

        c for c in NUMERIC_COLUMNS

        if any(
            pattern in c.lower()
            for pattern in patterns
        )

    ]


for group, columns in METRIC_COLUMNS.items():

    print(
        f"\n{group.upper()}"
    )
    print("-" * 100)

    for column in columns:
        print(column)


UTILITY_COLUMNS = METRIC_COLUMNS["utility"]

if UTILITY_COLUMNS:

    PRIMARY_UTILITY_COLUMN = (
        UTILITY_COLUMNS[0]
    )

else:

    PRIMARY_UTILITY_COLUMN = None


print("\nPrimary utility column:")
print(PRIMARY_UTILITY_COLUMN)

In [ ]:
# ============================================================
# NOTEBOOK 10.5 — METRIC DISCOVERY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.5")
print("METRIC DISCOVERY")
print("=" * 100)


NUMERIC_COLUMNS = [
    c for c in RESULTS.columns
    if pd.api.types.is_numeric_dtype(
        RESULTS[c]
    )
]


METRIC_PATTERNS = {

    "accuracy":
        ["mae", "rmse", "r2", "accuracy",
         "macro_f1", "weighted_f1"],

    "distribution":
        ["wasserstein", "js", "kl",
         "distribution"],

    "dependency":
        ["pearson", "spearman",
         "mutual_information",
         "dependency"],

    "predictive":
        ["predictive", "downstream",
         "auc", "f1"],

    "efficiency":
        ["runtime", "time", "memory",
         "cost"],

    "utility":
        ["utility", "score"]
}


METRIC_COLUMNS = {}


for group, patterns in METRIC_PATTERNS.items():

    METRIC_COLUMNS[group] = [

        c for c in NUMERIC_COLUMNS

        if any(
            pattern in c.lower()
            for pattern in patterns
        )

    ]


for group, columns in METRIC_COLUMNS.items():

    print(
        f"\n{group.upper()}"
    )
    print("-" * 100)

    for column in columns:
        print(column)


UTILITY_COLUMNS = METRIC_COLUMNS["utility"]

if UTILITY_COLUMNS:

    PRIMARY_UTILITY_COLUMN = (
        UTILITY_COLUMNS[0]
    )

else:

    PRIMARY_UTILITY_COLUMN = None


print("\nPrimary utility column:")
print(PRIMARY_UTILITY_COLUMN)

In [ ]:
# ============================================================
# NOTEBOOK 10.6 — SELECTED STRATEGY PERFORMANCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.6")
print("SELECTED STRATEGY PERFORMANCE")
print("=" * 100)


STRATEGY_COLUMNS = [
    c for c in RESULTS.columns
    if c.lower() in [
        "strategy",
        "selected_strategy",
        "selected_method",
        "method"
    ]
]


if STRATEGY_COLUMNS:

    STRATEGY_COLUMN = STRATEGY_COLUMNS[0]

    RESULTS["strategy"] = (
        RESULTS[STRATEGY_COLUMN]
        .astype(str)
    )

else:

    RESULTS["strategy"] = "AIR-LLM_SELECTED"


print(
    f"Strategy column : {STRATEGY_COLUMN}"
    if STRATEGY_COLUMNS
    else
    "Strategy column : not explicitly stored; "
    "using AIR-LLM_SELECTED"
)


if PRIMARY_UTILITY_COLUMN:

    RESULTS["utility"] = pd.to_numeric(
        RESULTS[PRIMARY_UTILITY_COLUMN],
        errors="coerce"
    )

else:

    RESULTS["utility"] = np.nan


SELECTED_RESULTS_PATH = (
    RESULTS_DIR /
    "selected_strategy_results.csv"
)

RESULTS.to_csv(
    SELECTED_RESULTS_PATH,
    index=False
)


display(
    RESULTS.head()
)

In [ ]:
# ============================================================
# NOTEBOOK 10.7 — CROSS-DATASET PERFORMANCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.7")
print("CROSS-DATASET PERFORMANCE")
print("=" * 100)


GROUP_COLUMNS = [
    "dataset_id",
    "mechanism",
    "requested_rate"
]


if RESULTS["utility"].notna().any():

    CROSS_DATASET_SUMMARY = (
        RESULTS
        .groupby(GROUP_COLUMNS, dropna=False)
        .agg(
            mean_utility=("utility", "mean"),
            std_utility=("utility", "std"),
            median_utility=("utility", "median"),
            min_utility=("utility", "min"),
            max_utility=("utility", "max"),
            observations=("utility", "count")
        )
        .reset_index()
    )

else:

    CROSS_DATASET_SUMMARY = (
        RESULTS
        .groupby(GROUP_COLUMNS, dropna=False)
        .size()
        .reset_index(name="observations")
    )


CROSS_DATASET_PATH = (
    ROBUSTNESS_DIR /
    "cross_dataset_performance.csv"
)

CROSS_DATASET_SUMMARY.to_csv(
    CROSS_DATASET_PATH,
    index=False
)


display(
    CROSS_DATASET_SUMMARY.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 10.9 — MECHANISM ROBUSTNESS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.9")
print("MISSINGNESS-MECHANISM ROBUSTNESS")
print("=" * 100)


if RESULTS["utility"].notna().any():

    MECHANISM_ROBUSTNESS = (
        RESULTS
        .groupby("mechanism")
        .agg(
            mean_utility=("utility", "mean"),
            std_utility=("utility", "std"),
            median_utility=("utility", "median"),
            minimum_utility=("utility", "min"),
            maximum_utility=("utility", "max"),
            observations=("utility", "count")
        )
        .reset_index()
    )

else:

    MECHANISM_ROBUSTNESS = (
        RESULTS
        .groupby("mechanism")
        .size()
        .reset_index(
            name="observations"
        )
    )


MECHANISM_ROBUSTNESS.to_csv(
    ROBUSTNESS_DIR /
    "mechanism_robustness.csv",
    index=False
)


display(
    MECHANISM_ROBUSTNESS
)

In [ ]:
# ============================================================
# NOTEBOOK 10.10 — MISSINGNESS RATE ROBUSTNESS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.10")
print("MISSINGNESS-RATE ROBUSTNESS")
print("=" * 100)


RATE_ROBUSTNESS = (
    RESULTS
    .groupby("requested_rate")
    .agg(
        mean_utility=("utility", "mean"),
        std_utility=("utility", "std"),
        median_utility=("utility", "median"),
        minimum_utility=("utility", "min"),
        maximum_utility=("utility", "max"),
        observations=("utility", "count")
    )
    .reset_index()
)


RATE_ROBUSTNESS.to_csv(
    ROBUSTNESS_DIR /
    "missingness_rate_robustness.csv",
    index=False
)


display(
    RATE_ROBUSTNESS
)

In [ ]:
# ============================================================
# NOTEBOOK 10.10 — MISSINGNESS RATE ROBUSTNESS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.10")
print("MISSINGNESS-RATE ROBUSTNESS")
print("=" * 100)


RATE_ROBUSTNESS = (
    RESULTS
    .groupby("requested_rate")
    .agg(
        mean_utility=("utility", "mean"),
        std_utility=("utility", "std"),
        median_utility=("utility", "median"),
        minimum_utility=("utility", "min"),
        maximum_utility=("utility", "max"),
        observations=("utility", "count")
    )
    .reset_index()
)


RATE_ROBUSTNESS.to_csv(
    ROBUSTNESS_DIR /
    "missingness_rate_robustness.csv",
    index=False
)


display(
    RATE_ROBUSTNESS
)

In [ ]:
# ============================================================
# NOTEBOOK 10.12 — WORST-CASE ROBUSTNESS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.12")
print("WORST-CASE ROBUSTNESS")
print("=" * 100)


WORST_CASE = (
    RESULTS
    .groupby("dataset_id")
    .agg(
        mean_utility=("utility", "mean"),
        worst_case_utility=("utility", "min"),
        best_case_utility=("utility", "max"),
        utility_std=("utility", "std")
    )
    .reset_index()
)


WORST_CASE[
    "robustness_gap"
] = (
    WORST_CASE["best_case_utility"]
    -
    WORST_CASE["worst_case_utility"]
)


WORST_CASE.to_csv(
    ROBUSTNESS_DIR /
    "worst_case_robustness.csv",
    index=False
)


display(
    WORST_CASE
)

In [ ]:
# ============================================================
# NOTEBOOK 10.12 — WORST-CASE ROBUSTNESS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.12")
print("WORST-CASE ROBUSTNESS")
print("=" * 100)


WORST_CASE = (
    RESULTS
    .groupby("dataset_id")
    .agg(
        mean_utility=("utility", "mean"),
        worst_case_utility=("utility", "min"),
        best_case_utility=("utility", "max"),
        utility_std=("utility", "std")
    )
    .reset_index()
)


WORST_CASE[
    "robustness_gap"
] = (
    WORST_CASE["best_case_utility"]
    -
    WORST_CASE["worst_case_utility"]
)


WORST_CASE.to_csv(
    ROBUSTNESS_DIR /
    "worst_case_robustness.csv",
    index=False
)


display(
    WORST_CASE
)

In [ ]:
# ============================================================
# NOTEBOOK 10.13 — STRATEGY SELECTION STABILITY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.13")
print("STRATEGY SELECTION STABILITY")
print("=" * 100)


if "strategy" in RESULTS.columns:

    STRATEGY_COUNTS = (
        RESULTS
        .groupby(
            [
                "dataset_id",
                "mechanism",
                "requested_rate",
                "strategy"
            ]
        )
        .size()
        .reset_index(
            name="selection_count"
        )
    )


    STRATEGY_COUNTS[
        "selection_probability"
    ] = (
        STRATEGY_COUNTS
        .groupby(
            [
                "dataset_id",
                "mechanism",
                "requested_rate"
            ]
        )["selection_count"]
        .transform(
            lambda x:
            x / x.sum()
        )
    )


    STRATEGY_STABILITY = (
        STRATEGY_COUNTS
        .groupby(
            [
                "dataset_id",
                "mechanism",
                "requested_rate"
            ]
        )["selection_probability"]
        .max()
        .reset_index(
            name="dominant_strategy_probability"
        )


    STRATEGY_STABILITY[
        "selection_stable"
    ] = (
        STRATEGY_STABILITY[
            "dominant_strategy_probability"
        ] >= 0.80
    )


else:

    STRATEGY_COUNTS = pd.DataFrame()
    STRATEGY_STABILITY = pd.DataFrame()


STRATEGY_COUNTS.to_csv(
    ROBUSTNESS_DIR /
    "strategy_selection_counts.csv",
    index=False
)

STRATEGY_STABILITY.to_csv(
    ROBUSTNESS_DIR /
    "strategy_selection_stability.csv",
    index=False
)


display(
    STRATEGY_STABILITY.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 10.14 — OVERALL ROBUSTNESS SCORE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.14")
print("OVERALL ROBUSTNESS")
print("=" * 100)


OVERALL_ROBUSTNESS = pd.DataFrame({

    "metric": [
        "datasets_evaluated",
        "mechanisms_evaluated",
        "missingness_rates_evaluated",
        "repetitions_evaluated",
        "total_result_rows"
    ],

    "value": [
        RESULTS["dataset_id"].nunique(),
        RESULTS["mechanism"].nunique(),
        RESULTS["requested_rate"].nunique(),
        RESULTS["repetition"].nunique(),
        len(RESULTS)
    ]

})


if RESULTS["utility"].notna().any():

    OVERALL_ROBUSTNESS = pd.concat(
        [
            OVERALL_ROBUSTNESS,
            pd.DataFrame({
                "metric": [
                    "overall_mean_utility",
                    "overall_std_utility",
                    "overall_median_utility",
                    "overall_min_utility",
                    "overall_max_utility"
                ],
                "value": [
                    RESULTS["utility"].mean(),
                    RESULTS["utility"].std(),
                    RESULTS["utility"].median(),
                    RESULTS["utility"].min(),
                    RESULTS["utility"].max()
                ]
            })
        ],
        ignore_index=True
    )


OVERALL_ROBUSTNESS.to_csv(
    ROBUSTNESS_DIR /
    "overall_robustness.csv",
    index=False
)


display(
    OVERALL_ROBUSTNESS
)

In [ ]:
# ============================================================
# NOTEBOOK 10.15 — STATISTICAL DISTRIBUTION ANALYSIS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.15")
print("STATISTICAL DISTRIBUTION ANALYSIS")
print("=" * 100)


from scipy import stats


STATISTICAL_SUMMARY = []


if RESULTS["utility"].notna().any():

    for dataset_id, group in RESULTS.groupby(
        "dataset_id"
    ):

        values = (
            group["utility"]
            .dropna()
            .to_numpy()
        )

        if len(values) < 3:
            continue

        row = {

            "dataset_id":
                dataset_id,

            "n":
                len(values),

            "mean":
                np.mean(values),

            "std":
                np.std(
                    values,
                    ddof=1
                ),

            "median":
                np.median(values),

            "q1":
                np.percentile(values, 25),

            "q3":
                np.percentile(values, 75),

            "min":
                np.min(values),

            "max":
                np.max(values)
        }

        if len(values) <= 5000:

            try:

                shapiro_stat, shapiro_p = (
                    stats.shapiro(values)
                )

                row["shapiro_stat"] = (
                    shapiro_stat
                )

                row["shapiro_p"] = (
                    shapiro_p
                )

            except Exception:

                row["shapiro_stat"] = np.nan
                row["shapiro_p"] = np.nan

        else:

            row["shapiro_stat"] = np.nan
            row["shapiro_p"] = np.nan

        STATISTICAL_SUMMARY.append(row)


STATISTICAL_SUMMARY_DF = pd.DataFrame(
    STATISTICAL_SUMMARY
)


STATISTICAL_SUMMARY_DF.to_csv(
    STATISTICS_DIR /
    "utility_distribution_summary.csv",
    index=False
)


display(
    STATISTICAL_SUMMARY_DF
)

In [ ]:
# ============================================================
# NOTEBOOK 10.16 — KRUSKAL-WALLIS DATASET TEST
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.16")
print("CROSS-DATASET KRUSKAL-WALLIS TEST")
print("=" * 100)


groups = []

dataset_names = []


for dataset_id, group in RESULTS.groupby(
    "dataset_id"
):

    values = (
        group["utility"]
        .dropna()
        .to_numpy()
    )

    if len(values) >= 2:

        groups.append(values)
        dataset_names.append(dataset_id)


if len(groups) >= 2:

    kw_stat, kw_p = (
        stats.kruskal(*groups)
    )

else:

    kw_stat = np.nan
    kw_p = np.nan


KRUSKAL_RESULT = pd.DataFrame({

    "test": [
        "Kruskal-Wallis"
    ],

    "groups": [
        " | ".join(dataset_names)
    ],

    "statistic": [
        kw_stat
    ],

    "p_value": [
        kw_p
    ],

    "significant_at_0.05": [
        bool(
            kw_p < 0.05
        )
        if not np.isnan(kw_p)
        else False
    ]
})


KRUSKAL_RESULT.to_csv(
    STATISTICS_DIR /
    "kruskal_wallis_dataset.csv",
    index=False
)


display(
    KRUSKAL_RESULT
)

In [ ]:
# ============================================================
# NOTEBOOK 10.17 — PAIRWISE DATASET COMPARISONS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.17")
print("PAIRWISE DATASET STATISTICAL COMPARISONS")
print("=" * 100)


from itertools import combinations


PAIRWISE_ROWS = []


dataset_groups = {

    dataset_id:
        group["utility"]
        .dropna()
        .to_numpy()

    for dataset_id, group
    in RESULTS.groupby("dataset_id")
}


for dataset_a, dataset_b in combinations(
    dataset_groups.keys(),
    2
):

    a = dataset_groups[dataset_a]
    b = dataset_groups[dataset_b]


    if len(a) >= 2 and len(b) >= 2:

        statistic, p_value = (
            stats.mannwhitneyu(
                a,
                b,
                alternative="two-sided"
            )
        )

    else:

        statistic = np.nan
        p_value = np.nan


    PAIRWISE_ROWS.append({

        "dataset_a":
            dataset_a,

        "dataset_b":
            dataset_b,

        "u_statistic":
            statistic,

        "p_value":
            p_value,

        "significant_at_0.05":
            (
                p_value < 0.05
                if not np.isnan(p_value)
                else False
            )
    })


PAIRWISE_DATASET_TESTS = pd.DataFrame(
    PAIRWISE_ROWS
)


PAIRWISE_DATASET_TESTS.to_csv(
    STATISTICS_DIR /
    "pairwise_dataset_tests.csv",
    index=False
)


display(
    PAIRWISE_DATASET_TESTS
)

In [ ]:
# ============================================================
# NOTEBOOK 10.18 — EFFECT SIZE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.18")
print("PAIRWISE EFFECT SIZE ANALYSIS")
print("=" * 100)


def cliffs_delta(x, y):

    x = np.asarray(x)
    y = np.asarray(y)

    x = x[~np.isnan(x)]
    y = y[~np.isnan(y)]

    if len(x) == 0 or len(y) == 0:
        return np.nan

    greater = 0
    less = 0

    for value_x in x:

        greater += np.sum(
            value_x > y
        )

        less += np.sum(
            value_x < y
        )

    return (
        greater - less
    ) / (
        len(x) * len(y)
    )


EFFECT_ROWS = []


for dataset_a, dataset_b in combinations(
    dataset_groups.keys(),
    2
):

    delta = cliffs_delta(
        dataset_groups[dataset_a],
        dataset_groups[dataset_b]
    )

    EFFECT_ROWS.append({

        "dataset_a":
            dataset_a,

        "dataset_b":
            dataset_b,

        "cliffs_delta":
            delta,

        "absolute_effect":
            abs(delta)
            if not np.isnan(delta)
            else np.nan
    })


EFFECT_SIZE_DF = pd.DataFrame(
    EFFECT_ROWS
)


EFFECT_SIZE_DF.to_csv(
    STATISTICS_DIR /
    "cliffs_delta_effect_sizes.csv",
    index=False
)


display(
    EFFECT_SIZE_DF
)

In [ ]:
# ============================================================
# NOTEBOOK 10.19 — MECHANISM STATISTICAL ANALYSIS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.19")
print("MISSINGNESS-MECHANISM STATISTICAL ANALYSIS")
print("=" * 100)


mechanism_groups = {

    mechanism:
        group["utility"]
        .dropna()
        .to_numpy()

    for mechanism, group
    in RESULTS.groupby("mechanism")
}


MECHANISM_TEST_ROWS = []


if len(mechanism_groups) >= 2:

    mechanism_values = list(
        mechanism_groups.values()
    )

    kw_stat, kw_p = (
        stats.kruskal(
            *mechanism_values
        )
    )

else:

    kw_stat = np.nan
    kw_p = np.nan


MECHANISM_TEST_ROWS.append({

    "test":
        "Kruskal-Wallis",

    "statistic":
        kw_stat,

    "p_value":
        kw_p,

    "significant_at_0.05":
        (
            kw_p < 0.05
            if not np.isnan(kw_p)
            else False
        )
})


MECHANISM_TEST_DF = pd.DataFrame(
    MECHANISM_TEST_ROWS
)


MECHANISM_TEST_DF.to_csv(
    STATISTICS_DIR /
    "mechanism_statistical_test.csv",
    index=False
)


display(
    MECHANISM_TEST_DF
)

In [ ]:
# ============================================================
# NOTEBOOK 10.21 — SCENARIO ROBUSTNESS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.21")
print("SCENARIO-LEVEL ROBUSTNESS")
print("=" * 100)


SCENARIO_COLUMNS = [
    "dataset_id",
    "mechanism",
    "requested_rate"
]


SCENARIO_ROBUSTNESS = (
    RESULTS
    .groupby(SCENARIO_COLUMNS)
    .agg(
        mean_utility=("utility", "mean"),
        std_utility=("utility", "std"),
        median_utility=("utility", "median"),
        minimum_utility=("utility", "min"),
        maximum_utility=("utility", "max"),
        repetitions=("repetition", "nunique")
    )
    .reset_index()
)


SCENARIO_ROBUSTNESS[
    "relative_variability"
] = (
    SCENARIO_ROBUSTNESS[
        "std_utility"
    ]
    /
    SCENARIO_ROBUSTNESS[
        "mean_utility"
    ].abs().replace(
        0,
        np.nan
    )
)


SCENARIO_ROBUSTNESS.to_csv(
    ROBUSTNESS_DIR /
    "scenario_robustness.csv",
    index=False
)


display(
    SCENARIO_ROBUSTNESS.head(25)
)

In [ ]:
# ============================================================
# NOTEBOOK 10.21 — SCENARIO ROBUSTNESS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.21")
print("SCENARIO-LEVEL ROBUSTNESS")
print("=" * 100)


SCENARIO_COLUMNS = [
    "dataset_id",
    "mechanism",
    "requested_rate"
]


SCENARIO_ROBUSTNESS = (
    RESULTS
    .groupby(SCENARIO_COLUMNS)
    .agg(
        mean_utility=("utility", "mean"),
        std_utility=("utility", "std"),
        median_utility=("utility", "median"),
        minimum_utility=("utility", "min"),
        maximum_utility=("utility", "max"),
        repetitions=("repetition", "nunique")
    )
    .reset_index()
)


SCENARIO_ROBUSTNESS[
    "relative_variability"
] = (
    SCENARIO_ROBUSTNESS[
        "std_utility"
    ]
    /
    SCENARIO_ROBUSTNESS[
        "mean_utility"
    ].abs().replace(
        0,
        np.nan
    )
)


SCENARIO_ROBUSTNESS.to_csv(
    ROBUSTNESS_DIR /
    "scenario_robustness.csv",
    index=False
)


display(
    SCENARIO_ROBUSTNESS.head(25)
)

In [ ]:
# ============================================================
# NOTEBOOK 10.22 — FAILURE AND INSTABILITY ANALYSIS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.22")
print("FAILURE / INSTABILITY ANALYSIS")
print("=" * 100)


FAILURE_ROWS = []


for dataset_id, group in RESULTS.groupby(
    "dataset_id"
):

    utility = group["utility"]

    FAILURE_ROWS.append({

        "dataset_id":
            dataset_id,

        "total_runs":
            len(group),

        "missing_utility":
            int(utility.isna().sum()),

        "valid_utility":
            int(utility.notna().sum()),

        "zero_utility":
            int(
                (utility == 0)
                .sum()
            ),

        "negative_utility":
            int(
                (utility < 0)
                .sum()
            )
    })


FAILURE_ANALYSIS = pd.DataFrame(
    FAILURE_ROWS
)


FAILURE_ANALYSIS[
    "failure_rate"
] = (
    FAILURE_ANALYSIS[
        "missing_utility"
    ]
    /
    FAILURE_ANALYSIS[
        "total_runs"
    ]
)


FAILURE_ANALYSIS.to_csv(
    ROBUSTNESS_DIR /
    "failure_analysis.csv",
    index=False
)


display(
    FAILURE_ANALYSIS
)

In [ ]:
# ============================================================
# NOTEBOOK 10.23 — PUBLICATION TABLES
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.23")
print("GENERATING PUBLICATION TABLES")
print("=" * 100)


# ------------------------------------------------------------
# Table 1 — Dataset robustness
# ------------------------------------------------------------

TABLE_1 = DATASET_ROBUSTNESS.copy()

TABLE_1.to_csv(
    TABLES_DIR /
    "Table_1_Dataset_Robustness.csv",
    index=False
)


# ------------------------------------------------------------
# Table 2 — Mechanism robustness
# ------------------------------------------------------------

TABLE_2 = MECHANISM_ROBUSTNESS.copy()

TABLE_2.to_csv(
    TABLES_DIR /
    "Table_2_Mechanism_Robustness.csv",
    index=False
)


# ------------------------------------------------------------
# Table 3 — Missingness-rate robustness
# ------------------------------------------------------------

TABLE_3 = RATE_ROBUSTNESS.copy()

TABLE_3.to_csv(
    TABLES_DIR /
    "Table_3_Missingness_Rate_Robustness.csv",
    index=False
)


# ------------------------------------------------------------
# Table 4 — Statistical tests
# ------------------------------------------------------------

TABLE_4 = pd.concat(
    [
        KRUSKAL_RESULT.assign(
            analysis="dataset"
        ),
        MECHANISM_TEST_DF.assign(
            analysis="mechanism"
        )
    ],
    ignore_index=True
)


TABLE_4.to_csv(
    TABLES_DIR /
    "Table_4_Statistical_Tests.csv",
    index=False
)


# ------------------------------------------------------------
# Table 5 — Effect sizes
# ------------------------------------------------------------

TABLE_5 = EFFECT_SIZE_DF.copy()

TABLE_5.to_csv(
    TABLES_DIR /
    "Table_5_Effect_Sizes.csv",
    index=False
)


print("\nPublication tables generated:")
print(f"Table 1 : {TABLES_DIR / 'Table_1_Dataset_Robustness.csv'}")
print(f"Table 2 : {TABLES_DIR / 'Table_2_Mechanism_Robustness.csv'}")
print(f"Table 3 : {TABLES_DIR / 'Table_3_Missingness_Rate_Robustness.csv'}")
print(f"Table 4 : {TABLES_DIR / 'Table_4_Statistical_Tests.csv'}")
print(f"Table 5 : {TABLES_DIR / 'Table_5_Effect_Sizes.csv'}")

In [ ]:
# ============================================================
# NOTEBOOK 10.24 — RESEARCH SUMMARY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.24")
print("RESEARCH SUMMARY")
print("=" * 100)


SUMMARY_ROWS = []


SUMMARY_ROWS.append({
    "analysis": "Datasets",
    "value": RESULTS["dataset_id"].nunique()
})


SUMMARY_ROWS.append({
    "analysis": "Mechanisms",
    "value": RESULTS["mechanism"].nunique()
})


SUMMARY_ROWS.append({
    "analysis": "Missingness rates",
    "value": RESULTS["requested_rate"].nunique()
})


SUMMARY_ROWS.append({
    "analysis": "Repetitions",
    "value": RESULTS["repetition"].nunique()
})


SUMMARY_ROWS.append({
    "analysis": "Experimental observations",
    "value": len(RESULTS)
})


if RESULTS["utility"].notna().any():

    SUMMARY_ROWS.extend([

        {
            "analysis":
                "Mean utility",

            "value":
                RESULTS["utility"].mean()
        },

        {
            "analysis":
                "Median utility",

            "value":
                RESULTS["utility"].median()
        },

        {
            "analysis":
                "Utility standard deviation",

            "value":
                RESULTS["utility"].std()
        },

        {
            "analysis":
                "Worst-case utility",

            "value":
                RESULTS["utility"].min()
        }
    ])


RESEARCH_SUMMARY = pd.DataFrame(
    SUMMARY_ROWS
)


RESEARCH_SUMMARY.to_csv(
    TABLES_DIR /
    "AIR_LLM_Final_Research_Summary.csv",
    index=False
)


display(
    RESEARCH_SUMMARY
)

In [ ]:
# ============================================================
# NOTEBOOK 10.25 — FINAL VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.25")
print("FINAL VALIDATION")
print("=" * 100)


VALIDATION = {}


# ------------------------------------------------------------
# Dataset validation
# ------------------------------------------------------------

VALIDATION[
    "dataset_coverage"
] = set(
    RESULTS["dataset_id"]
).issuperset(
    set(DATASETS)
)


# ------------------------------------------------------------
# Mechanism validation
# ------------------------------------------------------------

VALIDATION[
    "mechanism_coverage"
] = set(
    RESULTS["mechanism"]
).issuperset(
    set(MECHANISMS)
)


# ------------------------------------------------------------
# Rate validation
# ------------------------------------------------------------

observed_rates = set(
    np.round(
        RESULTS["requested_rate"]
        .dropna()
        .unique(),
        2
    )
)

expected_rates = set(
    np.round(
        MISSINGNESS_RATES,
        2
    )
)


VALIDATION[
    "rate_coverage"
] = (
    observed_rates
    .issuperset(
        expected_rates
    )
)


# ------------------------------------------------------------
# Repetition validation
# ------------------------------------------------------------

VALIDATION[
    "repetition_coverage"
] = (
    RESULTS["repetition"]
    .nunique()
    >= REPETITIONS
)


# ------------------------------------------------------------
# Utility validation
# ------------------------------------------------------------

VALIDATION[
    "utility_available"
] = (
    RESULTS["utility"]
    .notna()
    .any()
)


VALIDATION_DF = pd.DataFrame({

    "validation":
        list(
            VALIDATION.keys()
        ),

    "passed":
        list(
            VALIDATION.values()
        )
})


VALIDATION_DF.to_csv(
    METADATA_DIR /
    "validation.csv",
    index=False
)


display(
    VALIDATION_DF
)


if not all(
    VALIDATION.values()
):

    print(
        "\nWARNING: "
        "One or more validations did not pass."
    )

else:

    print(
        "\nALL NOTEBOOK 10 VALIDATIONS PASSED."
    )

In [ ]:
# ============================================================
# NOTEBOOK 10.26 — DRIVE PERSISTENCE VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.26")
print("DRIVE PERSISTENCE VALIDATION")
print("=" * 100)


REQUIRED_OUTPUTS = [

    ROBUSTNESS_DIR /
    "dataset_robustness.csv",

    ROBUSTNESS_DIR /
    "mechanism_robustness.csv",

    ROBUSTNESS_DIR /
    "missingness_rate_robustness.csv",

    ROBUSTNESS_DIR /
    "repetition_stability.csv",

    STATISTICS_DIR /
    "utility_distribution_summary.csv",

    STATISTICS_DIR /
    "kruskal_wallis_dataset.csv",

    STATISTICS_DIR /
    "pairwise_dataset_tests.csv",

    STATISTICS_DIR /
    "cliffs_delta_effect_sizes.csv",

    TABLES_DIR /
    "AIR_LLM_Final_Research_Summary.csv"
]


MISSING_OUTPUTS = [
    path
    for path in REQUIRED_OUTPUTS
    if not path.exists()
]


print(
    f"Required outputs : "
    f"{len(REQUIRED_OUTPUTS)}"
)

print(
    f"Missing outputs  : "
    f"{len(MISSING_OUTPUTS)}"
)


for path in MISSING_OUTPUTS:

    print(
        f"MISSING : {path}"
    )


if MISSING_OUTPUTS:

    raise RuntimeError(
        "Notebook 10 persistence validation failed."
    )


print(
    "\nDrive persistence validation PASSED."
)

In [ ]:
# ============================================================
# NOTEBOOK 10.27 — MANIFEST
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 10.27")
print("CREATING NOTEBOOK 10 MANIFEST")
print("=" * 100)


def file_hash(path):

    sha256 = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            sha256.update(chunk)

    return sha256.hexdigest()


output_files = []


for directory in [
    RESULTS_DIR,
    ROBUSTNESS_DIR,
    STATISTICS_DIR,
    TABLES_DIR
]:

    for path in directory.rglob("*"):

        if path.is_file():

            output_files.append({

                "file":
                    str(path.relative_to(
                        PROJECT_ROOT
                    )),

                "size_bytes":
                    path.stat().st_size,

                "sha256":
                    file_hash(path)
            })


MANIFEST = {

    "project":
        "AIR-LLM",

    "notebook":
        "10",

    "title":
        "Cross-Dataset Robustness + Statistical Analysis",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "configuration": {

        "datasets":
            DATASETS,

        "mechanisms":
            MECHANISMS,

        "missingness_rates":
            MISSINGNESS_RATES,

        "repetitions":
            REPETITIONS,

        "master_seed":
            MASTER_SEED
    },

    "result_rows":
        len(RESULTS),

    "outputs":
        output_files
}


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        MANIFEST,
        f,
        indent=2
    )


print(
    f"Manifest saved:\n{MANIFEST_PATH}"
)

print(
    f"Output files recorded: "
    f"{len(output_files)}"
)